# Angular 15 — Complete Reference Guide
### Detailed Notes | Examples | Use Cases | Interview Q&A

> **Angular 15** was released on **November 16, 2022**. It stabilized the **Standalone APIs**, introduced **Functional Router Guards**, the powerful **Directive Composition API**, the **`NgOptimizedImage`** directive for automatic image optimization, and dramatically improved error stack traces.

---

## Table of Contents
1. [Detailed Notes](#detailed-notes) — What changed and why
2. [Code Examples](#examples) — Hands-on code for every feature
3. [Use Cases](#use-cases) — Real-world application scenarios
4. [Interview Q&A](#interview-qa) — 20+ interview questions with answers

---

## Version Requirements

| Dependency | Required Version |
|---|---|
| Node.js | `^14.20.0 \|\| ^16.13.0 \|\| >=18.10.0` |
| TypeScript | `4.8.x` |
| RxJS | `^7.4.0` |
| Angular CLI | `15.x` |
| Zone.js | `~0.12.0` |

```bash
# Upgrade command
ng update @angular/core@15 @angular/cli@15
ng update @angular/material@15
```

# Section 1 — Detailed Notes

---

## 1.1 Stable Standalone APIs

Angular 15 graduates all Standalone APIs from **Developer Preview** to **Stable**. This means the API is production-ready, won't have breaking changes, and is the **recommended way** to build Angular apps going forward.

### What Became Stable
| API | Description |
|---|---|
| `standalone: true` | Component/Directive/Pipe decorator flag |
| `bootstrapApplication()` | Root bootstrap without NgModule |
| `provideRouter()` | Router configuration for standalone apps |
| `provideHttpClient()` | HTTP client for standalone apps |
| `provideAnimations()` | Animations for standalone apps |
| `importProvidersFrom()` | Bridge NgModule providers to standalone |
| `loadComponent` | Lazy load a single component via router |
| `loadChildren` (routes array) | Lazy load a routes array (no NgModule) |

### New Standalone `provide*` Functions in v15
```typescript
// main.ts — fully standalone app with all v15 stable providers
import { bootstrapApplication } from '@angular/platform-browser';
import { provideRouter, withPreloading, PreloadAllModules, withRouterConfig } from '@angular/router';
import { provideHttpClient, withInterceptors, withFetch } from '@angular/common/http';
import { provideAnimations } from '@angular/platform-browser/animations';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(
      routes,
      withPreloading(PreloadAllModules),
      withRouterConfig({ paramsInheritanceStrategy: 'always' })
    ),
    provideHttpClient(
      withFetch(),                        // use native Fetch API
      withInterceptors([authInterceptor]) // functional interceptors
    ),
    provideAnimations(),
  ]
});
```

---

## 1.2 Functional Router Guards

### The Problem with Class-Based Guards
Before Angular 15, every guard required a full class with `@Injectable`, constructor injection, and `implements CanActivate`. For simple logic, this was excessive boilerplate.

### Functional Guards — Plain Functions
Angular 15 allows guards to be **plain functions** (or arrow functions). The `inject()` function can be called inside them to get services.

```typescript
// Simple functional guard
export const authGuard: CanActivateFn = () => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  return auth.isLoggedIn() ? true : router.createUrlTree(['/login']);
};
```

### All Guard Types as Functions
| Interface | Functional Type |
|---|---|
| `CanActivate` | `CanActivateFn` |
| `CanActivateChild` | `CanActivateChildFn` |
| `CanDeactivate<T>` | `CanDeactivateFn<T>` |
| `CanMatch` | `CanMatchFn` |
| `Resolve<T>` | `ResolveFn<T>` |

```typescript
// CanDeactivate guard — protect unsaved changes
export const unsavedChangesGuard: CanDeactivateFn<EditComponent> = (component) => {
  if (component.form.dirty) {
    return confirm('You have unsaved changes. Leave anyway?');
  }
  return true;
};

// Resolve guard — prefetch data before route activates
export const userResolver: ResolveFn<User> = (route) => {
  return inject(UserService).getUser(route.paramMap.get('id')!);
};

// CanMatch guard — conditionally match routes
export const featureFlagGuard: CanMatchFn = () => {
  return inject(FeatureFlagService).isEnabled('new-dashboard');
};
```

### Route Definition with Functional Guards
```typescript
const routes: Routes = [
  {
    path: 'dashboard',
    component: DashboardComponent,
    canActivate: [authGuard],                    // functional guard
    canDeactivate: [unsavedChangesGuard],         // functional guard
    resolve: { user: userResolver }              // functional resolver
  },
  {
    path: 'beta-feature',
    component: BetaComponent,
    canMatch: [featureFlagGuard]                 // functional canMatch
  }
];
```

---

## 1.3 Directive Composition API

### What Is It?
The Directive Composition API allows you to **apply directives to a component via its metadata** using `hostDirectives`. This means behaviors from multiple directives can be composed into one component **without wrapping elements** or using mixins.

### Basic Composition
```typescript
// tooltip.directive.ts
@Directive({
  selector: '[appTooltip]',
  standalone: true
})
export class TooltipDirective {
  @Input('appTooltip') text = '';
  // tooltip display logic
}

// highlight.directive.ts
@Directive({
  selector: '[appHighlight]',
  standalone: true
})
export class HighlightDirective {
  @Input() highlightColor = 'yellow';
  @HostBinding('style.background') get bg() { return this.highlightColor; }
}

// button.component.ts — COMPOSE both directives
@Component({
  selector: 'app-button',
  standalone: true,
  hostDirectives: [
    HighlightDirective,          // apply as-is
    {
      directive: TooltipDirective,
      inputs:  ['appTooltip: tooltip'],  // rename input: directive input → component input
    }
  ],
  template: `<button><ng-content /></button>`
})
export class ButtonComponent {}
```

```html
<!-- Usage — both hostDirective behaviors available -->
<app-button tooltip="Save the document" highlightColor="#e0f7fa">
  Save
</app-button>
```

### Exposing Outputs from Host Directives
```typescript
@Directive({ selector: '[clickTracker]', standalone: true })
export class ClickTrackerDirective {
  @Output() tracked = new EventEmitter<MouseEvent>();

  @HostListener('click', ['$event'])
  onClick(e: MouseEvent) { this.tracked.emit(e); }
}

@Component({
  selector: 'app-card',
  standalone: true,
  hostDirectives: [{
    directive: ClickTrackerDirective,
    outputs: ['tracked: cardClicked']   // rename output for consumers
  }],
  template: `<div class="card"><ng-content /></div>`
})
export class CardComponent {}
```

```html
<!-- Consumer uses the exposed output name -->
<app-card (cardClicked)="onCardClick($event)">Content</app-card>
```

---

## 1.4 NgOptimizedImage Directive (Stable)

Angular 15 stabilizes `NgOptimizedImage` (moved from `@angular/common/experimental` to `@angular/common`).

### What It Does
- Sets `loading="lazy"` by default for off-screen images
- Sets `loading="eager"` + `fetchpriority="high"` for priority images (LCP)
- Warns at development time if `width` and `height` are missing
- Warns if an image causes layout shift
- Supports CDN image loaders for automatic `srcset` generation

### Basic Usage
```typescript
import { NgOptimizedImage } from '@angular/common';

@Component({
  standalone: true,
  imports: [NgOptimizedImage],
  template: `
    <!-- Standard image -->
    <img ngSrc="assets/hero.jpg" width="1200" height="600" alt="Hero banner">

    <!-- LCP image — loads eagerly, high fetch priority -->
    <img ngSrc="assets/banner.png" width="800" height="400" priority alt="Main banner">

    <!-- Fill mode — fills parent container (parent must have position: relative) -->
    <div style="position: relative; height: 300px;">
      <img ngSrc="assets/cover.jpg" fill alt="Cover photo">
    </div>
  `
})
export class HeroComponent {}
```

### Built-in CDN Loaders
```typescript
// main.ts — choose a CDN loader
import {
  provideImgixLoader,
  provideCloudinaryLoader,
  provideImageKitLoader,
  provideCloudflareLoader
} from '@angular/common';

bootstrapApplication(AppComponent, {
  providers: [
    // Imgix — automatically generates optimized srcset
    provideImgixLoader('https://mysite.imgix.net/'),

    // Cloudinary
    // provideCloudinaryLoader('https://res.cloudinary.com/my-account'),

    // ImageKit
    // provideImageKitLoader('https://ik.imagekit.io/my-id'),
  ]
});
```

```html
<!-- With CDN loader — srcset is generated automatically -->
<img ngSrc="profile/user-123.jpg" width="400" height="400" alt="Profile">
<!-- Generates: srcset="https://mysite.imgix.net/profile/user-123.jpg?w=400 1x,
                        https://mysite.imgix.net/profile/user-123.jpg?w=800 2x" -->
```

### Custom CDN Loader
```typescript
// custom-loader.ts
import { IMAGE_LOADER, ImageLoaderConfig } from '@angular/common';

export const myCustomLoader = {
  provide: IMAGE_LOADER,
  useValue: (config: ImageLoaderConfig) => {
    const { src, width, loaderParams } = config;
    const quality = loaderParams?.['quality'] ?? 80;
    return `https://my-cdn.com/${src}?w=${width}&q=${quality}&fmt=webp`;
  }
};
```

---

## 1.5 Improved Stack Traces

Angular 15 dramatically improves error messages by **filtering out Angular framework internals** from stack traces, showing only the app code that actually caused the error.

### Before Angular 15 (noisy)
```
Error: Cannot read properties of null
    at ZoneDelegate.invoke (zone.js:372)
    at Object.onInvoke (core.mjs:26390)
    at ZoneDelegate.invoke (zone.js:371)
    at Zone.run (zone.js:134)
    at NgZone.run (core.mjs:26186)
    at Anonymous (core.mjs:26327)
    ← 8 more framework frames...
    at UserComponent.loadUser (user.component.ts:42)   ← actual bug
```

### After Angular 15 (clean)
```
Error: Cannot read properties of null
    at UserComponent.loadUser (user.component.ts:42)   ← immediately visible
```

---

## 1.6 `provideRouter` — New Router Features

Angular 15 exposes previously hidden router features as **composable `with*` functions**:

```typescript
import {
  provideRouter,
  withPreloading,
  withDebugTracing,
  withEnabledBlockingInitialNavigation,
  withDisabledInitialNavigation,
  withRouterConfig,
  withHashLocation,
  withComponentInputBinding,   // bind route params to @Input() — v16 preview
  PreloadAllModules
} from '@angular/router';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(
      routes,
      withPreloading(PreloadAllModules),        // preload all lazy modules
      withDebugTracing(),                       // log router events (dev only)
      withHashLocation(),                       // use hash-based URLs (#/path)
      withRouterConfig({
        paramsInheritanceStrategy: 'always',   // child routes inherit parent params
        onSameUrlNavigation: 'reload'          // re-trigger guards on same URL
      }),
      withEnabledBlockingInitialNavigation()    // wait for first nav before rendering
    )
  ]
});
```

---

## 1.7 Angular Material — MDC-based Components (Stable)

Angular Material 15 migrates all components to **MDC (Material Design Components for Web)** which is the official Google implementation of Material Design.

### Key Changes
```typescript
// Theming — new density system
@use '@angular/material' as mat;

$my-theme: mat.define-light-theme((
  color: (
    primary:    mat.define-palette(mat.$indigo-palette),
    accent:     mat.define-palette(mat.$pink-palette),
    warn:       mat.define-palette(mat.$red-palette),
  ),
  typography: mat.define-typography-config(),
  density: 0   // 0 = default, -1 = compact, -2 = very compact, -3 = minimum
));

html { @include mat.all-component-themes($my-theme); }
```

### Migration from Legacy Components
```bash
# Automated migration from legacy to MDC components
ng generate @angular/material:mdc-migration

# Or migrate specific components
ng generate @angular/material:mdc-migration --components=button,card,input
```

---

## 1.8 TypeScript 4.8 Support

### Key TypeScript 4.8 Improvements in Angular Context

#### Improved Inference for Infer Types
```typescript
// Better type narrowing in conditional types
type NoUndefined<T> = T extends undefined ? never : T;

// TypeScript 4.8 — improved intersection type reduction
type Flatten<T> = T extends Array<infer Item> ? Item : T;
```

#### `--exactOptionalPropertyTypes`
```typescript
// tsconfig.json
{ "compilerOptions": { "exactOptionalPropertyTypes": true } }

// Now distinguishes between "property is undefined" vs "property not present"
interface Config {
  timeout?: number;   // with exactOptionalPropertyTypes:
                      // can be omitted, but cannot be set to undefined explicitly
}
```

# Section 2 — Code Examples

---

## Example 1: Complete Standalone App with All Stable APIs

```typescript
// src/main.ts
import { bootstrapApplication } from '@angular/platform-browser';
import { provideRouter, withPreloading, PreloadAllModules, withRouterConfig } from '@angular/router';
import { provideHttpClient, withInterceptors, withFetch } from '@angular/common/http';
import { provideAnimations } from '@angular/platform-browser/animations';
import { provideImgixLoader } from '@angular/common';
import { APP_ROUTES } from './app/app.routes';
import { authInterceptor } from './app/core/interceptors/auth.interceptor';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(
      APP_ROUTES,
      withPreloading(PreloadAllModules),
      withRouterConfig({ paramsInheritanceStrategy: 'always' })
    ),
    provideHttpClient(
      withFetch(),
      withInterceptors([authInterceptor])
    ),
    provideAnimations(),
    provideImgixLoader('https://myapp.imgix.net/'),
  ]
}).catch(err => console.error(err));
```

```typescript
// src/app/app.routes.ts
import { Routes } from '@angular/router';
import { authGuard, roleGuard } from './core/guards';
import { userResolver } from './core/resolvers';

export const APP_ROUTES: Routes = [
  {
    path: '',
    loadComponent: () => import('./features/home/home.component').then(m => m.HomeComponent),
    title: 'Home'
  },
  {
    path: 'profile/:id',
    loadComponent: () => import('./features/profile/profile.component').then(m => m.ProfileComponent),
    canActivate: [authGuard],
    resolve: { user: userResolver },
    title: 'Profile'
  },
  {
    path: 'admin',
    loadChildren: () => import('./features/admin/admin.routes').then(m => m.ADMIN_ROUTES),
    canActivate: [authGuard, roleGuard('admin')],
    title: 'Admin'
  },
  { path: '**', redirectTo: '' }
];
```

---

## Example 2: Functional Guards — Complete Implementation

```typescript
// core/guards/auth.guard.ts
import { inject } from '@angular/core';
import { CanActivateFn, Router } from '@angular/router';
import { AuthService } from '../services/auth.service';

// Simple auth check
export const authGuard: CanActivateFn = (route, state) => {
  const auth   = inject(AuthService);
  const router = inject(Router);

  if (auth.isLoggedIn()) return true;

  // Redirect to login, preserve the attempted URL
  return router.createUrlTree(['/login'], {
    queryParams: { returnUrl: state.url }
  });
};
```

```typescript
// core/guards/role.guard.ts
import { inject } from '@angular/core';
import { CanActivateFn, Router } from '@angular/router';
import { AuthService } from '../services/auth.service';

// Guard factory — returns a CanActivateFn configured for a specific role
export const roleGuard = (requiredRole: string): CanActivateFn => {
  return () => {
    const auth   = inject(AuthService);
    const router = inject(Router);
    const user   = auth.currentUser();

    if (user?.roles.includes(requiredRole)) return true;

    return router.createUrlTree(['/unauthorized']);
  };
};

// Usage:
// canActivate: [roleGuard('admin')]
// canActivate: [roleGuard('editor')]
```

```typescript
// core/guards/unsaved-changes.guard.ts
import { CanDeactivateFn } from '@angular/router';

// Generic unsaved changes guard — works with any component that has a dirty form
export interface CanDeactivateComponent {
  hasUnsavedChanges(): boolean;
}

export const unsavedChangesGuard: CanDeactivateFn<CanDeactivateComponent> =
  (component) => {
    if (!component.hasUnsavedChanges()) return true;
    return confirm('You have unsaved changes. Are you sure you want to leave?');
  };
```

```typescript
// core/resolvers/user.resolver.ts
import { inject } from '@angular/core';
import { ResolveFn } from '@angular/router';
import { catchError, EMPTY } from 'rxjs';
import { UserService } from '../services/user.service';
import { User } from '../models/user.model';

export const userResolver: ResolveFn<User> = (route) => {
  const userService = inject(UserService);
  const router      = inject(Router);
  const id = route.paramMap.get('id')!;

  return userService.getUser(id).pipe(
    catchError(() => {
      router.navigate(['/not-found']);
      return EMPTY;
    })
  );
};
```

```typescript
// feature/profile/profile.component.ts — using resolver data
@Component({
  standalone: true,
  imports: [CommonModule, ReactiveFormsModule],
  template: `
    <h1>{{ user.name }}</h1>
    <form [formGroup]="form" (ngSubmit)="save()">
      <input formControlName="bio">
      <button type="submit">Save</button>
    </form>
  `
})
export class ProfileComponent implements CanDeactivateComponent {
  private route = inject(ActivatedRoute);

  user: User = this.route.snapshot.data['user'];  // from resolver

  form = new FormGroup({
    bio: new FormControl<string>(this.user.bio ?? '', { nonNullable: true })
  });

  // Implements CanDeactivateComponent interface for the guard
  hasUnsavedChanges(): boolean {
    return this.form.dirty;
  }

  save() {
    this.form.markAsPristine();  // clears dirty flag after save
  }
}
```

---

## Example 3: Directive Composition API — Design System Components

```typescript
// directives/ripple.directive.ts
@Directive({ selector: '[appRipple]', standalone: true })
export class RippleDirective {
  @Input() rippleColor = 'rgba(0,0,0,0.1)';

  @HostListener('click', ['$event'])
  onClickCreate(event: MouseEvent) {
    const el = (event.currentTarget as HTMLElement);
    const ripple = document.createElement('span');
    ripple.style.cssText = `
      position: absolute; border-radius: 50%;
      background: ${this.rippleColor};
      width: 4px; height: 4px;
      animation: ripple 0.6s linear;
    `;
    el.appendChild(ripple);
    setTimeout(() => ripple.remove(), 600);
  }
}

// directives/tooltip.directive.ts
@Directive({ selector: '[appTooltip]', standalone: true })
export class TooltipDirective {
  @Input('appTooltip') tooltipText = '';
  private tooltipEl?: HTMLElement;

  @HostListener('mouseenter')
  show() {
    this.tooltipEl = document.createElement('div');
    this.tooltipEl.className = 'tooltip';
    this.tooltipEl.textContent = this.tooltipText;
    document.body.appendChild(this.tooltipEl);
  }

  @HostListener('mouseleave')
  hide() { this.tooltipEl?.remove(); }
}

// directives/disabled.directive.ts
@Directive({ selector: '[appDisabled]', standalone: true })
export class DisabledDirective {
  @Input('appDisabled') isDisabled = false;
  @HostBinding('attr.disabled') get disabled() { return this.isDisabled ? true : null; }
  @HostBinding('style.opacity') get opacity() { return this.isDisabled ? '0.5' : '1'; }
  @HostBinding('style.cursor')  get cursor()  { return this.isDisabled ? 'not-allowed' : 'pointer'; }
}
```

```typescript
// components/ui-button.component.ts — COMPOSED with all three directives
@Component({
  selector: 'ui-button',
  standalone: true,
  hostDirectives: [
    RippleDirective,      // always applied — adds click ripple
    {
      directive: TooltipDirective,
      inputs: ['appTooltip: tooltip']   // expose as 'tooltip'
    },
    {
      directive: DisabledDirective,
      inputs: ['appDisabled: disabled'] // expose as 'disabled'
    }
  ],
  template: `
    <button class="ui-btn" [class]="'ui-btn--' + variant">
      <ng-content />
    </button>
  `,
  styles: [`
    .ui-btn { position: relative; overflow: hidden; padding: 8px 16px; border: none; }
    .ui-btn--primary  { background: #6200ee; color: #fff; }
    .ui-btn--secondary{ background: transparent; border: 2px solid #6200ee; color: #6200ee; }
  `]
})
export class UiButtonComponent {
  @Input() variant: 'primary' | 'secondary' = 'primary';
}
```

```html
<!-- template — all three directives active on one component -->
<ui-button variant="primary" tooltip="Save your work" [disabled]="isSaving">
  Save
</ui-button>

<ui-button variant="secondary" tooltip="Discard all changes">
  Cancel
</ui-button>
```

---

## Example 4: NgOptimizedImage — E-Commerce Product Gallery

```typescript
// product-gallery.component.ts
import { Component, Input } from '@angular/core';
import { NgOptimizedImage } from '@angular/common';
import { NgFor } from '@angular/common';

interface Product {
  id: number;
  name: string;
  image: string;   // e.g. 'products/shoe-123.jpg'
  price: number;
}

@Component({
  selector: 'app-product-gallery',
  standalone: true,
  imports: [NgOptimizedImage, NgFor],
  template: `
    <!-- Hero/banner image — LCP element, load with high priority -->
    <div class="hero-banner">
      <img
        ngSrc="banners/summer-sale.jpg"
        width="1440"
        height="480"
        priority                          <!-- fetchpriority="high", eager load -->
        alt="Summer Sale — Up to 50% off"
      >
    </div>

    <!-- Product grid — lazy loaded images below the fold -->
    <div class="product-grid">
      <div class="product-card" *ngFor="let product of products">
        <img
          [ngSrc]="product.image"
          width="400"
          height="400"
          [alt]="product.name"
          <!-- lazy loading applied automatically for non-priority images -->
        >
        <h3>{{ product.name }}</h3>
        <p>{{ product.price | currency }}</p>
      </div>
    </div>

    <!-- Profile/avatar — fill mode -->
    <div class="avatar-container" style="position: relative; width: 80px; height: 80px;">
      <img ngSrc="users/avatar-001.jpg" fill alt="User avatar">
    </div>
  `
})
export class ProductGalleryComponent {
  @Input() products: Product[] = [];
}
```

```typescript
// main.ts — configure Cloudinary loader for this example
import { provideCloudinaryLoader } from '@angular/common';

bootstrapApplication(AppComponent, {
  providers: [
    provideCloudinaryLoader('https://res.cloudinary.com/my-store')
    // Image: 'products/shoe-123.jpg'
    // Generated URL: https://res.cloudinary.com/my-store/image/upload/f_auto,q_auto,w_400/products/shoe-123.jpg
    // Auto format (WebP/AVIF), auto quality, correct width
  ]
});
```

---

## Example 5: Functional HTTP Interceptor

```typescript
// core/interceptors/auth.interceptor.ts
import { HttpInterceptorFn, HttpRequest, HttpHandlerFn } from '@angular/common/http';
import { inject } from '@angular/core';
import { catchError, switchMap, throwError } from 'rxjs';
import { AuthService } from '../services/auth.service';
import { Router } from '@angular/router';

export const authInterceptor: HttpInterceptorFn = (req: HttpRequest<unknown>, next: HttpHandlerFn) => {
  const auth   = inject(AuthService);
  const router = inject(Router);

  const token = auth.getAccessToken();

  // Clone request and add Authorization header
  const authReq = token
    ? req.clone({ setHeaders: { Authorization: `Bearer ${token}` } })
    : req;

  return next(authReq).pipe(
    catchError(err => {
      if (err.status === 401) {
        // Token expired — try to refresh
        return auth.refreshToken().pipe(
          switchMap(newToken => {
            const retryReq = req.clone({
              setHeaders: { Authorization: `Bearer ${newToken}` }
            });
            return next(retryReq);
          }),
          catchError(() => {
            // Refresh failed — redirect to login
            auth.logout();
            router.navigate(['/login']);
            return throwError(() => err);
          })
        );
      }
      return throwError(() => err);
    })
  );
};
```

```typescript
// logging.interceptor.ts — another functional interceptor
export const loggingInterceptor: HttpInterceptorFn = (req, next) => {
  const start = Date.now();
  console.log(`[HTTP] ${req.method} ${req.url}`);

  return next(req).pipe(
    tap({
      next: res => {
        const elapsed = Date.now() - start;
        console.log(`[HTTP] ${req.method} ${req.url} → ${res.status} (${elapsed}ms)`);
      },
      error: err => console.error(`[HTTP ERROR] ${req.url}`, err)
    })
  );
};

// Combine multiple interceptors in main.ts
provideHttpClient(
  withFetch(),
  withInterceptors([loggingInterceptor, authInterceptor])
)
```

---

## Example 6: Advanced Directive Composition — Accessible Form Controls

```typescript
// directives/aria-required.directive.ts
@Directive({ selector: '[formControl],[formControlName]', standalone: true })
export class AriaRequiredDirective {
  @Input() required = false;
  @HostBinding('attr.aria-required') get ariaRequired() { return this.required || null; }
}

// directives/error-state.directive.ts
@Directive({ standalone: true })
export class ErrorStateDirective implements OnInit {
  private el = inject(ElementRef);
  private control?: AbstractControl;
  private ngControl = inject(NgControl, { optional: true, self: true });

  ngOnInit() { this.control = this.ngControl?.control ?? undefined; }

  @HostBinding('class.field--error')
  get hasError(): boolean {
    return !!(this.control?.invalid && this.control?.touched);
  }
}

// components/text-field.component.ts — compose both
@Component({
  selector: 'ui-text-field',
  standalone: true,
  imports: [ReactiveFormsModule],
  hostDirectives: [
    AriaRequiredDirective,
    ErrorStateDirective,
  ],
  template: `
    <div class="field-wrapper">
      <label>{{ label }}<span *ngIf="required" class="required">*</span></label>
      <input [formControl]="control" [type]="type" [placeholder]="placeholder">
      <span class="error-msg" *ngIf="control.invalid && control.touched">
        {{ getErrorMessage() }}
      </span>
    </div>
  `
})
export class TextFieldComponent {
  @Input({ required: true }) control!: FormControl;
  @Input() label = '';
  @Input() type: 'text' | 'email' | 'password' = 'text';
  @Input() placeholder = '';
  @Input() required = false;

  getErrorMessage(): string {
    if (this.control.hasError('required'))  return `${this.label} is required`;
    if (this.control.hasError('email'))     return 'Enter a valid email address';
    if (this.control.hasError('minlength')) {
      const min = this.control.errors!['minlength'].requiredLength;
      return `Minimum ${min} characters required`;
    }
    return 'Invalid value';
  }
}
```

---

## Example 7: Migrating Class-Based Guards to Functional

```typescript
// BEFORE — Angular 14 class-based guard (still works in v15, but deprecated)
@Injectable({ providedIn: 'root' })
export class AuthGuard implements CanActivate, CanActivateChild {
  constructor(private auth: AuthService, private router: Router) {}

  canActivate(route: ActivatedRouteSnapshot): boolean | UrlTree {
    return this.check();
  }

  canActivateChild(route: ActivatedRouteSnapshot): boolean | UrlTree {
    return this.check();
  }

  private check(): boolean | UrlTree {
    return this.auth.isLoggedIn()
      ? true
      : this.router.createUrlTree(['/login']);
  }
}

// AFTER — Angular 15 functional guards (recommended)
export const authGuard: CanActivateFn = () => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  return auth.isLoggedIn() ? true : router.createUrlTree(['/login']);
};

// canActivateChild reuses the same function!
export const authChildGuard: CanActivateChildFn = authGuard;

// In routes — usage is identical
{
  path: 'admin',
  canActivate:      [authGuard],
  canActivateChild: [authChildGuard],
}
```

# Section 3 — Use Cases

---

## Use Case 1: Healthcare Portal — Role-Based Routing with Functional Guards

**Problem:** A healthcare portal has four user roles: `patient`, `doctor`, `nurse`, and `admin`. Each role sees different routes and some routes require multiple conditions (logged in AND correct role AND active subscription).

**Solution with Angular 15 Functional Guards:**

```typescript
// guards/auth.guard.ts
export const authGuard: CanActivateFn = (route, state) => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  if (auth.isLoggedIn()) return true;
  return router.createUrlTree(['/login'], { queryParams: { returnUrl: state.url } });
};

// guards/role.guard.ts — composable guard factory
export const hasRole = (...roles: UserRole[]): CanActivateFn => () => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  const user   = auth.currentUser();
  if (user && roles.some(r => user.roles.includes(r))) return true;
  return router.createUrlTree(['/access-denied']);
};

// guards/subscription.guard.ts
export const subscriptionGuard: CanActivateFn = () => {
  const billing = inject(BillingService);
  const router  = inject(Router);
  return billing.hasActiveSubscription()
    ? true
    : router.createUrlTree(['/renew-subscription']);
};

// routes — compose multiple guards elegantly
export const PORTAL_ROUTES: Routes = [
  { path: 'dashboard',     loadComponent: () => import('./dashboard/dashboard.component').then(m => m.DashboardComponent),
    canActivate: [authGuard] },

  { path: 'patient-records', loadComponent: () => import('./records/records.component').then(m => m.RecordsComponent),
    canActivate: [authGuard, hasRole('doctor', 'nurse'), subscriptionGuard] },

  { path: 'prescriptions', loadComponent: () => import('./prescriptions/prescriptions.component').then(m => m.PrescriptionsComponent),
    canActivate: [authGuard, hasRole('doctor')] },

  { path: 'admin',         loadChildren: () => import('./admin/admin.routes').then(m => m.ADMIN_ROUTES),
    canActivate: [authGuard, hasRole('admin')] },

  { path: 'profile',       loadComponent: () => import('./profile/profile.component').then(m => m.ProfileComponent),
    canActivate: [authGuard],
    canDeactivate: [unsavedChangesGuard] }   // protect unsaved profile edits
];
```

**Benefits over class guards:**
- No `@Injectable`, no constructor — less boilerplate per guard
- `hasRole('doctor', 'nurse')` reads like English in route config
- Guard factory pattern allows role reuse across different routes
- Can compose 3 guards in `canActivate` array without creating 3 classes

---

## Use Case 2: News Media Site — Image Optimization with NgOptimizedImage

**Problem:** A news site has 50+ articles per page, each with a thumbnail. Core Web Vitals scores were poor: LCP (Largest Contentful Paint) was 4.8 seconds, and images caused layout shift (CLS score 0.25).

**Solution with Angular 15 NgOptimizedImage:**

```typescript
// article-list.component.ts
@Component({
  standalone: true,
  imports: [NgOptimizedImage, NgFor, DatePipe],
  template: `
    <!-- Featured article — LCP candidate, must be priority -->
    <article class="featured">
      <img
        [ngSrc]="featured.heroImage"
        width="1200"
        height="630"
        priority
        [alt]="featured.title"
      >
      <h1>{{ featured.title }}</h1>
    </article>

    <!-- Article grid — below the fold, lazy load -->
    <div class="article-grid">
      <article *ngFor="let article of articles" class="article-card">
        <img
          [ngSrc]="article.thumbnail"
          width="400"
          height="225"
          [alt]="article.title"
        >
        <h3>{{ article.title }}</h3>
        <time>{{ article.publishedAt | date:'mediumDate' }}</time>
      </article>
    </div>
  `
})
export class ArticleListComponent {
  @Input() featured!: Article;
  @Input() articles: Article[] = [];
}
```

```typescript
// main.ts — Cloudinary loader for automatic optimization
bootstrapApplication(AppComponent, {
  providers: [
    provideCloudinaryLoader('https://res.cloudinary.com/news-media')
    // Automatically generates:
    // - WebP/AVIF format
    // - Correct srcset for different screen densities
    // - Responsive sizes
  ]
});
```

**Results after migration:**
| Metric | Before | After |
|---|---|---|
| LCP | 4.8s | 1.6s |
| CLS | 0.25 | 0.02 |
| Total image bytes | 4.2MB | 780KB |
| Images lazy-loaded | 0 | 47/50 |

---

## Use Case 3: SaaS Platform — Directive Composition for Consistent UX

**Problem:** A project management SaaS has 30+ interactive UI components (buttons, chips, badges, links). Each needs: ripple effect on click, tooltip on hover, keyboard focus outline, and disabled state styling. Previously this required either:
- Duplicating code in every component, OR
- A messy base class approach

**Solution with Angular 15 Directive Composition API:**

```typescript
// The "interactive behavior" directives library

// 1. Focus ring for keyboard users (accessibility)
@Directive({ standalone: true })
export class FocusRingDirective {
  @HostBinding('class.focus-ring') isFocused = false;
  @HostListener('focusin')  onFocus() { this.isFocused = true; }
  @HostListener('focusout') onBlur()  { this.isFocused = false; }
}

// 2. Keyboard activation (Enter/Space triggers click)
@Directive({ standalone: true })
export class KeyboardActivationDirective {
  @HostListener('keydown.enter', ['$event'])
  @HostListener('keydown.space', ['$event'])
  onKey(e: KeyboardEvent) {
    e.preventDefault();
    (e.target as HTMLElement).click();
  }
}

// 3. Loading state
@Directive({ standalone: true })
export class LoadingStateDirective {
  @Input() loading = false;
  @HostBinding('attr.aria-busy') get ariaBusy() { return this.loading || null; }
  @HostBinding('class.loading')  get isLoading() { return this.loading; }
}

// Apply all interactive behaviors to EVERY button in the app
@Component({
  selector: 'pm-button',
  standalone: true,
  hostDirectives: [
    FocusRingDirective,
    KeyboardActivationDirective,
    { directive: RippleDirective,  inputs: ['rippleColor'] },
    { directive: TooltipDirective, inputs: ['appTooltip: tooltip'] },
    { directive: DisabledDirective, inputs: ['appDisabled: disabled'] },
    { directive: LoadingStateDirective, inputs: ['loading'] }
  ],
  template: `
    <button [attr.tabindex]="disabled ? -1 : 0" [class]="'pm-btn pm-btn--' + size">
      <span *ngIf="loading" class="spinner"></span>
      <ng-content *ngIf="!loading" />
    </button>
  `
})
export class PmButtonComponent {
  @Input() size: 'sm' | 'md' | 'lg' = 'md';
  @Input() disabled = false;
  @Input() loading = false;
}
```

```html
<!-- One component — six behaviors from composed directives -->
<pm-button
  size="md"
  tooltip="Save this project"
  [loading]="isSaving"
  [disabled]="form.invalid"
  (click)="saveProject()">
  Save Project
</pm-button>
```

**Savings:**
- 6 behaviors added via 5 lines of `hostDirectives` config
- Zero copy-paste across 30+ components
- Change `RippleDirective` once — all components updated

---

## Use Case 4: Enterprise App — Faster Debugging with Clean Stack Traces

**Problem:** A logistics enterprise app had frequent production errors. When the on-call team received Sentry error reports, each trace was 30+ lines of Angular internals before reaching the actual application code. Debugging took 2–3 hours per incident.

**Solution: Angular 15 Clean Stack Traces**

```typescript
// Before Angular 15 — Sentry alert shows:
Error: Cannot read properties of undefined (reading 'waybillNumber')
  at ZoneDelegate.invoke (zone.js:372:26)
  at Object.onInvoke (core.mjs:26390:33)
  at ZoneDelegate.invoke (zone.js:371:52)
  at Zone.run (zone.js:134:43)
  at NgZone.run (core.mjs:26186:28)
  at ComponentRef.detectChanges (core.mjs:26327:19)
  at ApplicationRef.tick (core.mjs:26511:34)
  at ApplicationRef._zone.onMicrotaskEmpty (core.mjs:26461:26)
  ← 6 more zone.js frames...
  at ShipmentComponent.loadShipment (shipment.component.ts:87:28)  ← ACTUAL BUG

// After Angular 15:
Error: Cannot read properties of undefined (reading 'waybillNumber')
  at ShipmentComponent.loadShipment (shipment.component.ts:87:28)  ← IMMEDIATELY VISIBLE
```

**Configuration for Sentry + Angular 15:**
```typescript
// sentry integration — works better with Angular 15 clean traces
import * as Sentry from '@sentry/angular';

Sentry.init({
  dsn: 'https://xxx@sentry.io/xxx',
  integrations: [
    new Sentry.BrowserTracing(),
    new Sentry.Replay()
  ],
  // Angular 15 already filters framework frames — no extra config needed!
});
```

**Impact:** Mean Time to Resolve (MTTR) reduced from 2.5 hours to 35 minutes.

---

## Use Case 5: Real Estate Portal — `canDeactivate` Guard for Multi-Step Forms

**Problem:** A property listing multi-step form had 4 steps (Basic Info → Media → Pricing → Preview). Users accidentally clicked "Back" and lost all their entered data.

**Solution with Angular 15 Functional `CanDeactivate` Guard:**

```typescript
// listing-form.routes.ts
import { unsavedChangesGuard } from '@core/guards';

export const LISTING_ROUTES: Routes = [
  {
    path: 'new',
    component: ListingFormShellComponent,
    canDeactivate: [unsavedChangesGuard],   // protects ALL steps
    children: [
      { path: 'basic',   component: BasicInfoStepComponent },
      { path: 'media',   component: MediaUploadStepComponent },
      { path: 'pricing', component: PricingStepComponent },
      { path: 'preview', component: PreviewStepComponent },
    ]
  }
];
```

```typescript
// listing-form-shell.component.ts
@Component({ standalone: true })
export class ListingFormShellComponent implements CanDeactivateComponent {
  private listingStore = inject(ListingStoreService);

  hasUnsavedChanges(): boolean {
    // Check if any step has entered data that hasn't been submitted
    return this.listingStore.hasPendingChanges();
  }
}

// unsaved-changes.guard.ts — reusable functional guard
export const unsavedChangesGuard: CanDeactivateFn<CanDeactivateComponent> = (component) => {
  if (!component.hasUnsavedChanges()) return true;

  // Show native confirm dialog (or replace with a custom dialog using RxJS)
  return confirm(
    'Your listing draft will be lost.\n\nAre you sure you want to leave?'
  );
};
```

---

## Use Cases Summary Table

| Use Case | Feature Used | Business Value |
|---|---|---|
| Healthcare Portal | Functional Guards + `hasRole()` factory | Clean role-based access, no boilerplate classes |
| News Media Site | `NgOptimizedImage` | LCP improved 66%, image bytes reduced 81% |
| SaaS Platform | Directive Composition API | 6 behaviors via metadata, zero code duplication |
| Logistics Enterprise | Clean Stack Traces | MTTR reduced from 2.5h to 35min |
| Real Estate Portal | `CanDeactivate` functional guard | Zero data loss in multi-step forms |

# Section 4 — Interview Q&A

---

## Basic Level Questions

---

### Q1. What are the most important features released in Angular 15?

**Answer:**
The five key features of Angular 15 are:
1. **Stable Standalone APIs** — `standalone: true`, `bootstrapApplication`, `provideRouter`, etc. are production-ready
2. **Functional Router Guards** — guards as plain functions using `inject()` instead of `@Injectable` classes
3. **Directive Composition API** — apply multiple directives to a component via `hostDirectives` in metadata
4. **`NgOptimizedImage` (Stable)** — automatic lazy loading, priority hints, CDN srcset generation
5. **Improved Stack Traces** — Angular framework frames hidden, only app code shown in errors

---

### Q2. What is a functional guard in Angular 15?

**Answer:**
A functional guard is a **plain function** (not a class) that implements route protection logic. It uses `inject()` to access services and returns `boolean | UrlTree | Observable<boolean | UrlTree>`.

```typescript
// Functional guard
export const authGuard: CanActivateFn = (route, state) => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  return auth.isLoggedIn() ? true : router.createUrlTree(['/login']);
};

// Route usage — same as class guard
{ path: 'dashboard', canActivate: [authGuard] }
```

Benefits: less code, no class/interface/decorator boilerplate, easily testable as pure functions.

---

### Q3. What is the Directive Composition API?

**Answer:**
The Directive Composition API allows a component to **include directive behaviors via metadata** using the `hostDirectives` property in `@Component`. This composes multiple directive behaviors into one component without template wrappers.

```typescript
@Component({
  selector: 'app-btn',
  standalone: true,
  hostDirectives: [
    RippleDirective,
    { directive: TooltipDirective, inputs: ['appTooltip: tooltip'] }
  ],
  template: `<button><ng-content /></button>`
})
export class BtnComponent {}
```

The `inputs` and `outputs` arrays in the config let you **rename** directive inputs/outputs when exposing them on the component.

---

### Q4. What does `NgOptimizedImage` do automatically?

**Answer:**
`NgOptimizedImage` (used via `ngSrc` attribute) automatically:

| Feature | Behavior |
|---|---|
| Lazy loading | Sets `loading="lazy"` for all non-priority images |
| Priority loading | Sets `loading="eager"` + `fetchpriority="high"` when `priority` is added |
| Size warnings | Warns in dev if `width`/`height` attributes are missing |
| CLS prevention | Prevents layout shift by enforcing dimensions |
| `srcset` generation | Works with CDN loaders to generate responsive image sets |
| `fill` mode | Fills parent container (replaces `width`/`height` with CSS) |

---

### Q5. What are the types of functional guards available in Angular 15?

**Answer:**

| Type | Functional Type | Purpose |
|---|---|---|
| `CanActivate` | `CanActivateFn` | Guard route activation |
| `CanActivateChild` | `CanActivateChildFn` | Guard child route activation |
| `CanDeactivate<T>` | `CanDeactivateFn<T>` | Guard route leaving (unsaved data) |
| `CanMatch` | `CanMatchFn` | Conditionally match a route |
| `Resolve<T>` | `ResolveFn<T>` | Pre-fetch data before route activates |

All use `inject()` for service access and share the same function signature pattern.

---

## Intermediate Level Questions

---

### Q6. How do you expose inputs and outputs from `hostDirectives`?

**Answer:**
By default, directive inputs/outputs applied via `hostDirectives` are **not visible** to parent components. You must explicitly expose them using the `inputs` and `outputs` arrays with optional renaming:

```typescript
@Component({
  hostDirectives: [
    {
      directive: TooltipDirective,
      inputs: ['appTooltip: tooltip'],   // directive input 'appTooltip' → component input 'tooltip'
    },
    {
      directive: ClickTrackerDirective,
      outputs: ['tracked: itemClicked']  // directive output 'tracked' → component output 'itemClicked'
    }
  ]
})
export class CardComponent {}
```

```html
<!-- Consumer sees the renamed versions -->
<app-card tooltip="Click card" (itemClicked)="handle($event)"></app-card>
```

---

### Q7. How do you write unit tests for a functional guard?

**Answer:**
Functional guards are **plain functions** — they're much easier to test than class-based guards. You use `TestBed.runInInjectionContext()` to run the guard in an injection context:

```typescript
describe('authGuard', () => {
  let authService: jasmine.SpyObj<AuthService>;
  let router: Router;

  beforeEach(() => {
    TestBed.configureTestingModule({
      providers: [
        Router,
        { provide: AuthService, useValue: jasmine.createSpyObj('AuthService', ['isLoggedIn']) }
      ]
    });
    authService = TestBed.inject(AuthService) as jasmine.SpyObj<AuthService>;
    router = TestBed.inject(Router);
  });

  it('returns true when logged in', () => {
    authService.isLoggedIn.and.returnValue(true);
    const result = TestBed.runInInjectionContext(() => authGuard(fakeRoute, fakeState));
    expect(result).toBe(true);
  });

  it('redirects to /login when not logged in', () => {
    authService.isLoggedIn.and.returnValue(false);
    const result = TestBed.runInInjectionContext(() => authGuard(fakeRoute, fakeState));
    expect(result).toEqual(router.createUrlTree(['/login']));
  });
});
```

---

### Q8. What are CDN image loaders in Angular 15 and how do you create a custom one?

**Answer:**
CDN loaders are provider functions that tell `NgOptimizedImage` how to build the final image URL from a source name. Angular 15 ships four built-in loaders:

```typescript
provideImgixLoader('https://myapp.imgix.net/')
provideCloudinaryLoader('https://res.cloudinary.com/myaccount')
provideImageKitLoader('https://ik.imagekit.io/my-id')
provideCloudflareLoader('https://myapp.pages.dev')
```

**Custom loader:**
```typescript
import { IMAGE_LOADER, ImageLoaderConfig } from '@angular/common';

const myCDNLoader = {
  provide: IMAGE_LOADER,
  useValue: (config: ImageLoaderConfig): string => {
    const { src, width, loaderParams } = config;
    const quality = loaderParams?.['quality'] ?? 85;
    const format  = loaderParams?.['format']  ?? 'webp';
    return `https://my-cdn.com/img/${src}?w=${width}&q=${quality}&fmt=${format}`;
  }
};

// Usage in template with loaderParams
// <img ngSrc="photo.jpg" width="400" height="300"
//      [loaderParams]="{ quality: 95, format: 'avif' }">
```

---

### Q9. What is the difference between `CanActivate` and `CanMatch` guards?

**Answer:**

| | `CanActivate` / `CanActivateFn` | `CanMatch` / `CanMatchFn` |
|---|---|---|
| When it runs | After route is matched | Before route is matched |
| On false | Shows component but blocks navigation | Route is skipped, tries next matching route |
| URL change | URL changes to blocked route | URL stays on current route |
| Use case | Auth check — redirect to login | Feature flags — fall through to another route |

```typescript
const routes: Routes = [
  // canMatch — if false, Angular tries NEXT matching route
  {
    path: 'dashboard',
    component: NewDashboardComponent,
    canMatch: [() => inject(FeatureFlagService).isEnabled('new-dashboard')]
  },
  {
    path: 'dashboard',     // fallback if canMatch above returns false
    component: OldDashboardComponent
  }
];
```

---

### Q10. How does `NgOptimizedImage`'s `priority` attribute improve Core Web Vitals?

**Answer:**
The LCP (Largest Contentful Paint) metric measures when the largest visible element loads. For image-heavy pages, the hero/banner image is usually the LCP element.

Without `priority`:
- Image gets `loading="lazy"` → browser defers it → LCP is slow

With `priority`:
```html
<img ngSrc="banner.jpg" width="1200" height="400" priority alt="Banner">
```
Angular generates:
```html
<img src="banner.jpg" loading="eager" fetchpriority="high"
     width="1200" height="400" alt="Banner">
```

- `fetchpriority="high"` tells the browser to download this image **before** other non-critical resources
- `loading="eager"` means it's not lazy-loaded
- Result: LCP typically improves by 1–3 seconds on image-heavy pages

Angular also adds a `<link rel="preload">` to the `<head>` for priority images in SSR mode.

---

## Advanced Level Questions

---

### Q11. Explain the internal mechanism of Directive Composition API. When are host directives instantiated?

**Answer:**
Host directives are **instantiated as part of the component's host element**. They share the same host element and injector as the component, but have their own lifecycle.

**Instantiation order:**
1. Host directives are created **before** the component itself
2. Their `ngOnInit` runs before the component's `ngOnInit`
3. They are destroyed when the component is destroyed

**Key rules:**
- Host directives must be **standalone**
- A host directive can itself have `hostDirectives` (chaining)
- They **cannot** inject the host component (no circular DI)
- They share the host element's DOM — `@HostBinding` applies to the same element

```typescript
// Host directive can use @HostBinding on the component's host element
@Directive({ standalone: true })
export class DisabledDirective {
  @Input() disabled = false;
  // This @HostBinding applies to the COMPONENT's host <app-button> element
  @HostBinding('attr.disabled') get attr() { return this.disabled ? '' : null; }
}
```

---

### Q12. How do you compose the same directive behavior across different component types in a design system?

**Answer:**
Create a **"behavior directive"** and compose it via `hostDirectives` in all relevant components:

```typescript
// shared directive
@Directive({ standalone: true })
export class InteractiveDirective {
  @Input()  disabled = false;
  @Input()  loading  = false;
  @Output() activate = new EventEmitter<void>();

  @HostBinding('attr.aria-disabled') get ariaDisabled() { return this.disabled || null; }
  @HostBinding('attr.aria-busy')     get ariaBusy()     { return this.loading || null; }

  @HostListener('click')
  onClick() { if (!this.disabled && !this.loading) this.activate.emit(); }

  @HostListener('keydown.enter')
  @HostListener('keydown.space', ['$event'])
  onKey(e?: KeyboardEvent) { e?.preventDefault(); this.onClick(); }
}

// Apply to Button, Chip, Badge, Card, MenuItem — all get identical behavior
@Component({
  selector: 'ds-button',
  hostDirectives: [{ directive: InteractiveDirective, inputs: ['disabled','loading'], outputs: ['activate'] }],
  template: `<button><ng-content /></button>`
})
export class DsButtonComponent {}

@Component({
  selector: 'ds-chip',
  hostDirectives: [{ directive: InteractiveDirective, inputs: ['disabled'], outputs: ['activate'] }],
  template: `<span class="chip"><ng-content /></span>`
})
export class DsChipComponent {}
```

---

### Q13. What is `withComponentInputBinding()` introduced in Angular 15 and what does it do?

**Answer:**
`withComponentInputBinding()` (added in Angular 15, stable in Angular 16) enables **automatic binding of route data to component `@Input()` properties**:

```typescript
// Enable in providers
provideRouter(routes, withComponentInputBinding())
```

```typescript
// Route
{ path: 'user/:id', component: UserComponent, resolve: { user: userResolver } }

// Component — BEFORE (inject ActivatedRoute manually)
export class UserComponent implements OnInit {
  constructor(private route: ActivatedRoute) {}
  ngOnInit() {
    this.id   = this.route.snapshot.paramMap.get('id');
    this.user = this.route.snapshot.data['user'];
  }
}

// Component — AFTER (Angular 15+ with withComponentInputBinding)
export class UserComponent {
  @Input() id!: string;     // ← from :id route param
  @Input() user!: User;     // ← from resolve: { user: resolver }
  @Input() tab?: string;    // ← from ?tab= query param
}
```

Eliminates the need to inject `ActivatedRoute` for common data access patterns.

---

### Q14. How do functional resolvers handle errors differently than class-based resolvers?

**Answer:**
Functional resolvers use RxJS operators directly — error handling is done inline:

```typescript
// Class-based resolver — error handling via catchError
@Injectable()
export class ProductResolver implements Resolve<Product> {
  resolve(route: ActivatedRouteSnapshot): Observable<Product> {
    return this.productService.getProduct(route.paramMap.get('id')!).pipe(
      catchError(() => { this.router.navigate(['/not-found']); return EMPTY; })
    );
  }
}

// Functional resolver — same logic, less boilerplate
export const productResolver: ResolveFn<Product> = (route) => {
  const productService = inject(ProductService);
  const router         = inject(Router);

  return productService.getProduct(route.paramMap.get('id')!).pipe(
    catchError(() => { router.navigate(['/not-found']); return EMPTY; })
  );
};
```

Both approaches behave identically. If the resolver returns `EMPTY`, the navigation is cancelled.

---

### Q15. What problems does `NgOptimizedImage`'s `fill` mode solve?

**Answer:**
`fill` mode is used when you **don't know the image dimensions in advance** — for example, avatars, background images, or images in fluid containers. Without `fill`, you must specify `width` and `height`.

```html
<!-- fill mode — image stretches to fill parent container -->
<div style="position: relative; width: 100%; height: 300px;">
  <img ngSrc="hero.jpg" fill alt="Hero">
</div>
```

Angular generates:
```html
<img src="hero.jpg" style="position: absolute; width: 100%; height: 100%;"
     loading="lazy" alt="Hero">
```

**Problems it solves:**
1. **Unknown dimensions**: No need to hardcode `width`/`height` when container size varies
2. **Responsive images**: Image fills whatever space its container provides
3. **Avoids layout shift**: Container has explicit dimensions (via CSS), image fills it — no CLS

**Rule:** The parent container **must** have `position: relative` (or `absolute`/`fixed`) and explicit dimensions.

---

## Scenario-Based Questions

---

### Q16. A project has 20 class-based guards across 50 routes. How would you migrate to functional guards in Angular 15?

**Answer:**
**Strategy: Replace one guard at a time, starting with the simplest.**

```bash
# Step 1: Identify all guards
grep -r "implements CanActivate" src/ --include="*.ts"
```

```typescript
// Step 2: Create functional equivalent alongside the class guard
// OLD guard (keep temporarily)
@Injectable({ providedIn: 'root' })
export class AuthGuard implements CanActivate {
  canActivate(): boolean | UrlTree { /* ... */ }
}

// NEW functional guard (add in same file or new file)
export const authGuard: CanActivateFn = () => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  return auth.isLoggedIn() ? true : router.createUrlTree(['/login']);
};

// Step 3: Update routes to use functional guard
// canActivate: [AuthGuard]  →  canActivate: [authGuard]

// Step 4: Remove class guard after all routes are updated
// Step 5: Remove from providers (if it was explicitly provided)
```

**Priority order for migration:**
1. Simple boolean guards first
2. Guards with 1 dependency
3. Complex guards with multiple services last

---

### Q17. An image on your Angular 15 page fails to display, and you see a console warning: "NG02952: The NgOptimizedImage directive has detected that the image with src `hero.jpg` does not have a width or height." How do you fix it?

**Answer:**
`NgOptimizedImage` **requires** `width` and `height` attributes (except in `fill` mode) to calculate the image aspect ratio and prevent Cumulative Layout Shift.

**Fix options:**

```html
<!-- Option 1: Add explicit width and height (preferred) -->
<img ngSrc="hero.jpg" width="1200" height="630" alt="Hero">

<!-- Option 2: Use fill mode if dimensions are unknown -->
<div style="position: relative; height: 400px;">
  <img ngSrc="hero.jpg" fill alt="Hero">
</div>

<!-- Option 3: Aspect ratio via CSS + fill -->
<div style="position: relative; aspect-ratio: 16/9;">
  <img ngSrc="hero.jpg" fill style="object-fit: cover;" alt="Hero">
</div>
```

**Also check:** `width` and `height` should match the image's **intrinsic size**, not the displayed size. CSS handles the display scaling.

---

### Q18. How do you test a `CanDeactivate` functional guard?

**Answer:**
```typescript
// unsaved-changes.guard.spec.ts
import { TestBed } from '@angular/core/testing';
import { unsavedChangesGuard, CanDeactivateComponent } from './unsaved-changes.guard';

describe('unsavedChangesGuard', () => {
  const fakeComponent = (hasChanges: boolean): CanDeactivateComponent => ({
    hasUnsavedChanges: () => hasChanges
  });

  beforeEach(() => { TestBed.configureTestingModule({}); });

  it('returns true when no unsaved changes', () => {
    const result = TestBed.runInInjectionContext(() =>
      unsavedChangesGuard(fakeComponent(false), null!, null!, null!)
    );
    expect(result).toBe(true);
  });

  it('calls confirm when there are unsaved changes', () => {
    spyOn(window, 'confirm').and.returnValue(false);

    const result = TestBed.runInInjectionContext(() =>
      unsavedChangesGuard(fakeComponent(true), null!, null!, null!)
    );

    expect(window.confirm).toHaveBeenCalled();
    expect(result).toBe(false);
  });
});
```

---

### Q19. Your design system needs a `FocusTrap` behavior on all modal components. How would you use Directive Composition API for this?

**Answer:**
```typescript
// focus-trap.directive.ts
@Directive({ standalone: true })
export class FocusTrapDirective implements OnInit, OnDestroy {
  private el = inject(ElementRef<HTMLElement>);
  private focusableElements: HTMLElement[] = [];

  ngOnInit() {
    this.focusableElements = Array.from(
      this.el.nativeElement.querySelectorAll(
        'button, [href], input, select, textarea, [tabindex]:not([tabindex="-1"])'
      )
    );
    this.focusableElements[0]?.focus();   // focus first element on open
  }

  @HostListener('keydown.tab', ['$event'])
  @HostListener('keydown.shift.tab', ['$event'])
  onTab(event: KeyboardEvent) {
    const first = this.focusableElements[0];
    const last  = this.focusableElements[this.focusableElements.length - 1];

    if (event.shiftKey && document.activeElement === first) {
      last.focus(); event.preventDefault();
    } else if (!event.shiftKey && document.activeElement === last) {
      first.focus(); event.preventDefault();
    }
  }

  ngOnDestroy() { /* restore focus to trigger element */ }
}

// Apply to EVERY modal component via hostDirectives
@Component({
  selector: 'ui-modal',
  standalone: true,
  hostDirectives: [FocusTrapDirective],   // ← one line — all modals get focus trap
  template: `
    <div class="modal" role="dialog" aria-modal="true">
      <ng-content />
    </div>
  `
})
export class UiModalComponent {}
```

---

### Q20. What will happen if you use `loading="lazy"` manually on an image that also uses `ngSrc` with `priority`?

**Answer:**
It will cause a **compile-time/runtime warning** from `NgOptimizedImage`. The `priority` attribute is specifically designed to set `loading="eager"` and `fetchpriority="high"`. Setting `loading="lazy"` conflicts with this.

```html
<!-- ❌ Angular will warn: conflicting attributes -->
<img ngSrc="banner.jpg" width="1200" height="400" priority loading="lazy" alt="Banner">

<!-- ✅ Correct — let NgOptimizedImage manage loading behavior -->
<img ngSrc="banner.jpg" width="1200" height="400" priority alt="Banner">
<!-- Angular outputs: loading="eager" fetchpriority="high" automatically -->

<!-- ✅ For non-priority images — lazy is default, no need to add it -->
<img ngSrc="thumbnail.jpg" width="400" height="300" alt="Thumbnail">
<!-- Angular outputs: loading="lazy" automatically -->
```

The warning exists because manually overriding loading attributes defeats the purpose of `NgOptimizedImage` managing performance automatically.

---

## Quick Reference Card

### Angular 15 Key Facts for Interviews

| Question | Answer |
|---|---|
| Release date | November 16, 2022 |
| Biggest feature | Stable Standalone APIs — production-ready |
| Guard style | Functional guards (class guards deprecated) |
| Directive behavior reuse | Directive Composition API (`hostDirectives`) |
| Image optimization | `NgOptimizedImage` — stable in `@angular/common` |
| Stack traces | Angular framework frames filtered out |
| HTTP interceptors | Functional (`HttpInterceptorFn`) |
| TypeScript version | 4.8.x |
| CDN Loaders | Imgix, Cloudinary, ImageKit, Cloudflare (built-in) |
| `provideRouter` features | `withPreloading`, `withDebugTracing`, `withHashLocation`, `withComponentInputBinding` |

### Before vs After Cheatsheet

```
Class-based CanActivate          →  CanActivateFn (plain function + inject())
Class-based CanDeactivate<T>     →  CanDeactivateFn<T>
Class-based Resolve<T>           →  ResolveFn<T>
Class-based HttpInterceptor      →  HttpInterceptorFn
Mixin/base class for directives  →  hostDirectives composition
<img loading="lazy">             →  <img ngSrc="..." width="X" height="Y">
<img src="hero.jpg">  (LCP)      →  <img ngSrc="hero.jpg" width="X" height="Y" priority>
Developer Preview Standalone     →  Stable Standalone (v15)
Noisy 30-line stack traces       →  Clean 1-3 line stack traces
```